In [1]:
import pandas as pd
import numpy as np

fos = pd.read_csv(
    "../data/raw/fos/Most-Recent-Cohorts-Field-of-Study.csv",
    usecols=["UNITID", "CIPCODE", "CIPDESC", "CREDLEV", "CREDDESC", "IPEDSCOUNT1", "IPEDSCOUNT2", "DISTANCE"],
    na_values=["PrivacySuppressed", "NULL"], low_memory=False,
)
schools = pd.read_csv("../data/processed/schools_with_sevp.csv")

fos = fos[fos["CREDLEV"].isin([2, 3])].copy()        # 2 = associate's, 3 = bachelor's: the degrees our clients pursue
print("Rows with no UNITID (can't link to a school, dropped):", fos["UNITID"].isna().sum())
fos = fos.dropna(subset=["UNITID"])
fos["UNITID"] = fos["UNITID"].astype(int)
fos = fos[fos["UNITID"].isin(schools["unit_id"])]    # only schools in our universe
# Note: DISTANCE = 3 means a program CAN be completed fully online, not that it's online-only,
# so it can't identify programs F-1 students are barred from. Online-only units are handled in the SEVP step.
# CIPCODE is stored as an integer with leading zeros dropped: 5207 = 52.07, 109 = 01.09. Pad back to 4 digits.
fos["cip4"] = fos["CIPCODE"].astype(int).astype(str).str.zfill(4)
fos["cip2"] = fos["cip4"].str[:2]                    # 2-digit family, e.g. "52" = business
print("Undergrad degree program rows:", fos.shape)
print(fos["CREDDESC"].value_counts().to_string())
print("\nCIP 2401 description (expected: Liberal Arts / General Studies):",
      fos.loc[fos["cip4"] == "2401", "CIPDESC"].unique())

Rows with no UNITID (can't link to a school, dropped): 2320
Undergrad degree program rows: (105715, 10)
CREDDESC
Bachelor's Degree     67254
Associate's Degree    38461

CIP 2401 description (expected: Liberal Arts / General Studies): ['Liberal Arts and Sciences, General Studies and Humanities.']


In [2]:
# The same program can repeat across rows (e.g. by delivery mode) -> keep one row per school x program x level
prog = fos.drop_duplicates(["UNITID", "cip4", "CREDLEV"])

def cip_list(level):
    # For each school: its 4-digit program codes at this level, joined like "55207;5214"
    return prog[prog["CREDLEV"] == level].groupby("UNITID")["cip4"].agg(lambda s: ";".join(sorted(set(s))))

schools["bachelor_cips"] = schools["unit_id"].map(cip_list(3)).fillna("")
schools["associate_cips"] = schools["unit_id"].map(cip_list(2)).fillna("")
# General transfer track at a JUCO = Liberal Arts & Sciences / General Studies associate degree (CIP 24.01)
schools["has_transfer_track"] = schools["associate_cips"].str.split(";").apply(lambda cs: "2401" in cs)

def offers(cips, prefix):
    # True if any program code starts with the prefix ("52" = any business, "5207" = entrepreneurship)
    return any(c.startswith(prefix) for c in cips.split(";") if c)

schools["offers_entrepreneurship"] = schools["bachelor_cips"].apply(lambda s: offers(s, "5207")) | \
                                     schools["associate_cips"].apply(lambda s: offers(s, "5207"))

# Schools with no undergrad program data: unknown, not failing. The matcher keeps them.
schools["programs_known"] = (schools["bachelor_cips"] != "") | (schools["associate_cips"] != "")

In [3]:
# Validation
b4 = schools["school_type"] == "4-year"
print("4-year schools with a business bachelor's (52.xx):",
      schools[b4]["bachelor_cips"].apply(lambda s: offers(s, "52")).sum(), "of", b4.sum())
print("2-year schools with a business associate (52.xx):",
      schools[~b4]["associate_cips"].apply(lambda s: offers(s, "52")).sum(), "of", (~b4).sum())
print("2-year schools with a transfer track (24.01):", schools[~b4]["has_transfer_track"].sum(), "of", (~b4).sum())
print("2-year schools with NO associate programs listed at all:", (schools[~b4]["associate_cips"] == "").sum())
print("\nSchools offering entrepreneurship (52.07), by type:")
print(schools.groupby("school_type")["offers_entrepreneurship"].sum().to_string())

check = schools[schools["name"].str.contains("Washburn University|Neosho County|Garden City Community|Western Texas College",
                                             regex=True, na=False)]
print("\nSpot check:")
for _, r in check.iterrows():
    print(f"- {r['name']}: business bachelor's={offers(r['bachelor_cips'], '52')}, "
          f"business associate={offers(r['associate_cips'], '52')}, transfer track={r['has_transfer_track']}, "
          f"entrepreneurship={r['offers_entrepreneurship']}")

schools.to_csv("../data/processed/schools_with_majors.csv", index=False)
print("\nSaved:", schools.shape, "-> data/processed/schools_with_majors.csv")

4-year schools with a business bachelor's (52.xx): 1467 of 1797
2-year schools with a business associate (52.xx): 1051 of 1350
2-year schools with a transfer track (24.01): 994 of 1350
2-year schools with NO associate programs listed at all: 133

Schools offering entrepreneurship (52.07), by type:
school_type
2-year    200
4-year    312

Spot check:
- Garden City Community College: business bachelor's=False, business associate=True, transfer track=True, entrepreneurship=False
- Neosho County Community College: business bachelor's=False, business associate=True, transfer track=True, entrepreneurship=False
- Washburn University: business bachelor's=True, business associate=True, transfer track=True, entrepreneurship=True
- Western Texas College: business bachelor's=False, business associate=True, transfer track=True, entrepreneurship=False

Saved: (3147, 59) -> data/processed/schools_with_majors.csv


In [4]:
# Save a cip4 → program-name lookup for the app's school detail view.
# fos is still in scope from cell 0; strip trailing periods from CIPDESC.
cip_names = (
    fos[["cip4", "CIPDESC"]]
    .drop_duplicates("cip4")
    .assign(CIPDESC=lambda df: df["CIPDESC"].str.rstrip("."))
    .rename(columns={"CIPDESC": "name"})
    .sort_values("cip4")
    .reset_index(drop=True)
)
cip_names.to_csv("../data/processed/cip_names.csv", index=False)
print("CIP names saved:", cip_names.shape)
print(cip_names.head(10).to_string())

CIP names saved: (406, 2)
   cip4                                                      name
0  0100                                      Agriculture, General
1  0101                      Agricultural Business and Management
2  0102                                Agricultural Mechanization
3  0103                        Agricultural Production Operations
4  0104                 Agricultural and Food Products Processing
5  0105                 Agricultural and Domestic Animal Services
6  0106  Applied Horticulture and Horticultural Business Services
7  0107                                 International Agriculture
8  0108                              Agricultural Public Services
9  0109                                           Animal Sciences
